In [ ]:
# 1. Configure this notebook before using Run All (Google Colab or local Jupyter).
# Current PyPI release is 0.1.1; these v0.2.0 examples default to a pinned Git commit.
NSB_SOURCE = "github"  # "github" or "pypi"
NSB_VERSION = "0.2.0"  # exact nsb-py release in PyPI mode (must be published)
NSB_GIT_REF = "3b0832418a91576b0b3c20fec1f31c0446761059"  # exact VPRamon/NSB commit for GitHub mode
INSTALL_DEPENDENCIES = True  # False only for an already prepared, verified kernel


# CTAO sky-background API example

Use the same NSB public API at two representative CTAO locations. These approximate
coordinates and planning profiles illustrate geographic configuration; they do not
supply an instrument calibration. Radiance is in photons cm⁻² ns⁻¹ sr⁻¹.

Run the configuration cell first, then **Run All**. Setup installs the official `nsb-py` distribution (imported as `nsb`) and required plotting/astronomy packages. Choose an exact PyPI release or a remote GitHub commit; no local checkout is required, including in Google Colab. A GitHub build needs Rust 1.89+ (installed automatically on Linux when missing). Restart the kernel before changing sources. This notebook needs no datasets or internal helper package.

**v0.2.0:** Automatic resolves to PALACE Oxygen, with **Experimental** maturity,
standard continuum geometry, and **zenith distance ≤60°**. The Paranal-derived
model is a spatial extrapolation at other sites. Invalid Oxygen geometry raises
an error; no continuum fallback occurs. Explicit continuum preserves historical behavior.

For independent-observation evidence and limitations, consult NSB's
[scientific validation guide](../docs/scientific-validation.md).
Patat informed development; LVM absolute equivalence remains pending.
These examples are not calibrated camera rates or validated CTAO predictions.

Code: SPDX-License-Identifier: AGPL-3.0-only. Copyright (C) 2026 Vallés Puig, Ramon.


In [ ]:
# Install from PyPI or a remote GitHub revision (works without a local checkout).
import importlib.metadata as _metadata
import json as _json
import os as _os
from pathlib import Path as _Path
import re as _re
import shutil as _shutil
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
from urllib.request import urlretrieve as _urlretrieve

if NSB_SOURCE not in ("pypi", "github"):
    raise ValueError("NSB_SOURCE must be 'pypi' or 'github'")
if "nsb" in _sys.modules:
    raise RuntimeError("Restart the kernel before running setup again or changing NSB_SOURCE")

if NSB_SOURCE == "github":
    if not _re.fullmatch(r"[0-9a-f]{40}", NSB_GIT_REF):
        raise ValueError("Set NSB_GIT_REF to a full 40-character commit SHA for reproducibility")
    if INSTALL_DEPENDENCIES:
        _rust = _shutil.which("rustc")
        _rust_info = _subprocess.run([_rust, "--version"], capture_output=True, text=True).stdout if _rust else ""
        _match = _re.search(r"rustc (\d+)\.(\d+)", _rust_info)
        if not _match or tuple(map(int, _match.groups())) < (1, 89):
            if _sys.platform != "linux":
                raise RuntimeError("Building NSB from GitHub requires Rust 1.89+; install rustup first")
            print("Installing/updating the official Rust toolchain for the source build...")
            if _shutil.which("rustup"):
                _subprocess.run(["rustup", "default", "stable"], check=True)
                _subprocess.run(["rustup", "update", "stable"], check=True)
            else:
                with _tempfile.TemporaryDirectory() as _tmp:
                    _script = _Path(_tmp) / "rustup-init.sh"
                    _urlretrieve("https://sh.rustup.rs", _script)
                    _subprocess.run(["sh", str(_script), "-y", "--profile", "minimal"], check=True)
            _os.environ["PATH"] = str(_Path.home() / ".cargo/bin") + _os.pathsep + _os.environ["PATH"]
    _package = f"nsb-py @ git+https://github.com/VPRamon/NSB.git@{NSB_GIT_REF}"
else:
    _package = f"nsb-py=={NSB_VERSION}"

if INSTALL_DEPENDENCIES:
    _subprocess.run(
        [_sys.executable, "-m", "pip", "install", "--quiet", _package, "astropy", "numpy", "matplotlib"],
        check=True,
    )
else:
    print("Package installation skipped (INSTALL_DEPENDENCIES=False).")

_dist = _metadata.distribution("nsb-py")
if NSB_SOURCE == "pypi" and _dist.version != NSB_VERSION:
    raise RuntimeError(f"Expected nsb-py=={NSB_VERSION}, found {_dist.version}")
_direct = _json.loads(_dist.read_text("direct_url.json") or "{}")
if NSB_SOURCE == "github":
    if (_direct.get("url") != "https://github.com/VPRamon/NSB.git"
            or _direct.get("vcs_info", {}).get("commit_id") != NSB_GIT_REF):
        raise RuntimeError("The installed nsb-py is not from the selected NSB GitHub commit")
elif _direct.get("vcs_info"):
    raise RuntimeError("Expected a PyPI release, but nsb-py came from Git; restart and reinstall")

import nsb as _nsb
_files = {_Path(_dist.locate_file(p)).resolve() for p in (_dist.files or [])}
if _Path(_nsb.__file__).resolve() not in _files:
    raise RuntimeError("Import 'nsb' is shadowed by a different package; inspect/uninstall the conflicting distribution")
if not hasattr(_nsb, "NsbEvaluator") or not hasattr(_nsb, "NsbModelConfig"):
    raise RuntimeError("The installed 'nsb' does not expose the official NSB Python API")
print(f"NSB ready: nsb-py {_dist.version}, source={NSB_SOURCE}"
      + (f", commit={NSB_GIT_REF}" if NSB_SOURCE == "github" else ""))


In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Copyright (C) 2026 Vallés Puig, Ramon
from datetime import datetime, timezone
import astropy.units as u
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.time import Time
from astropy.utils import iers
import matplotlib.pyplot as plt
import numpy as np
import nsb
import siderust

iers.conf.auto_download = False
instant = datetime(2024, 1, 15, 2, tzinfo=timezone.utc)
sites = {
    "CTAO South": (-70.404, -24.627, 2150, nsb.SiteProfile.CTA_SOUTH),
    "CTAO North": (-17.890, 28.762, 2200, nsb.SiteProfile.CTA_NORTH),
}

def direction_at(location, zd):
    # Transform actual local altitude to ICRS, without refraction (pressure=0).
    sky = SkyCoord(az=0*u.deg, alt=(90-zd)*u.deg,
                   frame=AltAz(obstime=Time(instant), location=location, pressure=0*u.hPa)).icrs
    return siderust.Direction(ra_deg=float(sky.ra.deg), dec_deg=float(sky.dec.deg))


## Automatic, Oxygen and continuum

Hold solar flux and site configuration fixed. The default and explicit Oxygen
must agree exactly; metadata preserves the requested selection. Fixing F10.7
here is a reproducible input choice, not a site calibration.


In [ ]:
evaluators, geometry = {}, {}
for name, (lon, lat, height, profile) in sites.items():
    observer = siderust.Observer(lon_deg=lon, lat_deg=lat, height_m=height)
    location = EarthLocation.from_geodetic(lon*u.deg, lat*u.deg, height*u.m)
    geometry[name] = (observer, location)
    config = (nsb.NsbModelConfig.generic_clear_sky()
              .with_site_profile(profile).with_solar_radio_flux_sfu(130))
    models = {
        "Automatic": nsb.NsbEvaluator(config),
        "Oxygen": nsb.NsbEvaluator(config.with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_OXYGEN_V1)),
        "Continuum": nsb.NsbEvaluator(config.with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_V1)),
    }
    evaluators[name] = models
    query = nsb.PointQuery(observer, instant, direction_at(location, 0), components=nsb.ComponentMask.AIRGLOW)
    values = {key: evaluator.integrate_band(query, 300, 650) for key, evaluator in models.items()}
    assert values["Automatic"] == values["Oxygen"]
    assert values["Oxygen"] > values["Continuum"]
    for key in ("Automatic", "Oxygen"):
        metadata = models[key].evaluate(query).components[0].metadata
        print(name, key, metadata.airglow_selection_kind, metadata.airglow_resolved_model)
    print(name, values)


## Components and zenith distance

The left panel separates public sky radiance components at local zenith. The right
panel compares Airglow models within the Oxygen domain. The Moon contribution
reflects the actual geometry at this instant. These are illustrative conditions,
not an observing-window recommendation or detector simulation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
zenith_distances = np.linspace(0, 59, 16)  # Leave margin for frame transformations at 60°.
for index, (name, models) in enumerate(evaluators.items()):
    observer, location = geometry[name]
    query = nsb.PointQuery(observer, instant, direction_at(location, 0))
    components = models["Automatic"].evaluate(query).components
    axes[0].bar(np.arange(len(components)) + index*0.38,
                [c.integrated_photons_cm2_ns_sr for c in components], width=0.38, label=name)
    axes[0].set_xticks(np.arange(len(components))+0.19, [c.name for c in components])
    for label in ("Automatic", "Continuum"):
        values = [models[label].integrate_band(
            nsb.PointQuery(observer, instant, direction_at(location, float(zd)),
                           components=nsb.ComponentMask.AIRGLOW), 300, 650)
                  for zd in zenith_distances]
        axes[1].plot(zenith_distances, values, label=f"{name}: {label}")
axes[0].set_title("Sky components at zenith (300–650 nm)")
axes[1].set_title("Airglow (300–650 nm)")
axes[1].set_xlabel("Zenith distance (degrees)")
for ax in axes:
    ax.set_ylabel("photons cm⁻² ns⁻¹ sr⁻¹")
    ax.legend(fontsize=8)
plt.show()


## Explicit domain errors

Planning tracks and sky scans can leave the Oxygen domain. Restrict such queries
to ZD ≤60°, or deliberately select continuum for broader coverage. The latter
changes the physical model and must be recorded in any comparison.


In [ ]:
for name, models in evaluators.items():
    observer, location = geometry[name]
    query = nsb.PointQuery(observer, instant, direction_at(location, 70), components=nsb.ComponentMask.AIRGLOW)
    for label in ("Automatic", "Oxygen"):
        try:
            models[label].integrate_band(query, 300, 650)
        except nsb.OutOfRangeError as error:
            print(name, label, str(error))
        else:
            raise AssertionError("Out-of-domain Oxygen unexpectedly succeeded")
    assert np.isfinite(models["Continuum"].integrate_band(query, 300, 650))
